# IntentSight — Connection Readiness Pipeline

**WID3006 ML Group Assignment: "Tying the (Data) Knot"**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iztzx/WID3006_ML/blob/main/IntentSight_Colab.ipynb)

## Problem Statement

Modern relationships are increasingly shaped by digital interactions — swipe patterns, message frequency, emoji usage, and online presence all leave behavioral traces. This project tackles a **multi-class classification** problem: **can we predict a user's connection-readiness stage from their dating-app behavioral signals and demographic profile?**

We define five connection-readiness stages using a composite `connection_score` (weighted blend of match quality, conversation quality, profile quality, and activity level, minus swipe excess). Users are ranked by percentile and assigned to stages based on funnel-informed thresholds. The ML pipeline trains, tunes, and compares six classifiers, applies SHAP for interpretability, and calibrates the winning model for reliable probability estimates.

**Target:** `connection_stage` — five classes:

| Stage | Selection Criteria |
|---|---|
| **Likely To Connect** | `connection_score` rank >= 0.80 |
| **Ready To Chat** | Default middle tier |
| **Mostly Browsing** | Middle tier + `browser_issue` >= 62nd pctl + `browser_issue` >= `swipe_issue` |
| **Swipes Too Freely** | Middle tier + `swipe_issue` >= 50th pctl |
| **Needs Profile Help** | `connection_score` rank <= 0.20 |

**Dataset:** [Dating App Behavior Dataset](https://www.kaggle.com/datasets/keyushnisar/dating-app-behavior-dataset) — 50,000 synthetic records with 19 features spanning demographics, app usage, and match outcomes.

## Pipeline Overview

1. Install dependencies
2. Upload dataset
3. Exploratory Data Analysis (EDA)
4. Connection scoring and feature engineering (inline)
5. Preprocessing and target construction
6. Train 6 models with 5-fold cross-validation
7. Hyperparameter tuning on top 3
8. Select best model, calibrate, and validate
9. SHAP interpretability
10. Final comparison and classification report
11. AutoML baseline comparison
12. Save artifacts

**Runtime:** ~10-15 minutes on T4 GPU

## 1. Install Dependencies

In [1]:
!pip install -q pandas numpy scikit-learn imbalanced-learn xgboost lightgbm catboost shap matplotlib seaborn scipy joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00:00:0100:01m


## 2. Upload Dataset

Upload `Behaviour_Extended_Dataset.csv` from your local machine, **or** it will be auto-loaded if present.

In [ ]:
from pathlib import Path

DATASET_PATH = Path("Behaviour_Extended_Dataset.csv")

if not DATASET_PATH.exists():
    print("Upload Behaviour_Extended_Dataset.csv:")
    from google.colab import files

    uploaded = files.upload()
    print(f"Uploaded: {list(uploaded.keys())}")
else:
    print(f"Dataset found: {DATASET_PATH}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
matplotlib.use("Agg")

# Load raw data for EDA
eda_df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {eda_df.shape}")
print(f"\nColumn types:\n{eda_df.dtypes}")
print(f"\nMissing values per column:\n{eda_df.isnull().sum()}")
print(f"\nBasic statistics:\n{eda_df.describe().round(2)}")

In [ ]:
# --- 1. Gender and Income distributions ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if "gender" in eda_df.columns:
    sns.countplot(x="gender", data=eda_df, palette="Set3", ax=axes[0])
    axes[0].set_title("Gender Distribution")

if "income_bracket" in eda_df.columns:
    income_order = [
        "Very Low",
        "Low",
        "Lower-Middle",
        "Middle",
        "Upper-Middle",
        "High",
        "Very High",
    ]
    sns.countplot(
        x="income_bracket",
        data=eda_df,
        order=income_order,
        palette="coolwarm",
        ax=axes[1],
    )
    axes[1].set_title("Income Bracket Distribution")
    axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# --- 2. Behavioral feature distributions ---
behav_cols = [
    "app_usage_time_min",
    "swipe_right_ratio",
    "message_sent_count",
    "likes_received",
    "emoji_usage_rate",
    "mutual_matches",
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(behav_cols):
    row, col_idx = divmod(idx, 3)
    if col in eda_df.columns:
        sns.histplot(
            eda_df[col], bins=40, kde=True, ax=axes[row][col_idx], color="steelblue"
        )
        axes[row][col_idx].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# --- 3. Correlation heatmap of numeric features ---
numeric_cols = eda_df.select_dtypes(include=[np.number]).columns.tolist()
corr = eda_df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    cmap="RdBu_r",
    center=0,
    annot=False,
    fmt=".2f",
    square=True,
    linewidths=0.5,
)
plt.title("Correlation Heatmap of Numeric Features", fontsize=14)
plt.tight_layout()
plt.show()

# --- 4. Interest tags analysis ---
if "interest_tags" in eda_df.columns:
    tags = eda_df["interest_tags"].fillna("").str.split(r",\s*")
    tag_counts = tags.explode().value_counts()
    tag_counts = tag_counts[tag_counts.index != ""].head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=tag_counts.values, y=tag_counts.index, palette="viridis")
    plt.title("Top 15 Interest Tags", fontsize=14)
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()

print(f"\nEDA complete — {len(eda_df)} records, {eda_df.shape[1]} columns")
print(f"Missing values: {eda_df.isnull().sum().sum()}")

## 4. Connection Scoring and Feature Engineering

This section inlines the logic from `connection_scoring.py` — the composite scoring system and feature engineering pipeline. All functions are self-contained.

In [ ]:
# ===========================================================================
# Connection Scoring — inline from connection_scoring.py
# ===========================================================================

TARGET_COL = "connection_stage"

STAGE_LABELS = [
    "Needs Profile Help",
    "Mostly Browsing",
    "Swipes Too Freely",
    "Ready To Chat",
    "Likely To Connect",
]


def _numeric(frame, column, default=0.0):
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce").fillna(default)


def _rank01(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    if values.nunique(dropna=False) <= 1:
        return pd.Series(0.5, index=values.index, dtype="float64")
    return values.rank(pct=True).clip(0, 1)


def _bounded(series, cap):
    values = pd.to_numeric(series, errors="coerce").fillna(0).clip(lower=0)
    return (values / cap).clip(0, 1)


def _num_interests(frame):
    if "interest_tags" in frame.columns:
        tags = frame["interest_tags"].fillna("").astype(str).str.split(r",\s*")
        return tags.apply(lambda values: len([tag for tag in values if tag]))
    if "num_interests" in frame.columns:
        return _numeric(frame, "num_interests")
    return pd.Series(0, index=frame.index, dtype="float64")


def add_connection_features(frame):
    """Add interpretable product features used by the connection-readiness model."""
    engineered = frame.copy()

    app_usage = _numeric(engineered, "app_usage_time_min")
    swipe_ratio = _numeric(engineered, "swipe_right_ratio")
    likes = _numeric(engineered, "likes_received")
    matches = _numeric(engineered, "mutual_matches")
    messages = _numeric(engineered, "message_sent_count")
    emoji_rate = _numeric(engineered, "emoji_usage_rate")
    bio_length = _numeric(engineered, "bio_length")
    profile_pics = _numeric(engineered, "profile_pics_count")
    height = _numeric(engineered, "height_cm")
    weight = _numeric(engineered, "weight_kg")

    engineered["num_interests"] = _num_interests(engineered)
    engineered["match_rate"] = matches / (likes + 1)
    engineered["msg_per_match"] = messages / (matches + 1)
    engineered["bmi"] = np.where(height > 0, weight / ((height / 100) ** 2), 0)
    engineered["profile_completeness"] = (
        profile_pics.clip(0, 6) / 6 * 0.40
        + bio_length.clip(0, 300) / 300 * 0.40
        + engineered["num_interests"].clip(0, 5) / 5 * 0.20
    )
    engineered["selectivity_balance"] = (1 - (swipe_ratio - 0.55).abs() / 0.55).clip(
        0, 1
    )
    engineered["swipe_excess"] = (swipe_ratio - 0.70).clip(lower=0)
    engineered["like_to_match_gap"] = (likes - matches).clip(lower=0)
    engineered["conversation_depth"] = np.log1p(messages) * np.log1p(
        engineered["msg_per_match"]
    )
    engineered["social_pull"] = likes / (profile_pics + 1)
    engineered["activity_level"] = np.log1p(app_usage)

    if "last_active_hour" in engineered.columns:
        hour = _numeric(engineered, "last_active_hour") % 24
        engineered["last_active_sin"] = np.sin(2 * np.pi * hour / 24)
        engineered["last_active_cos"] = np.cos(2 * np.pi * hour / 24)

    engineered["match_quality"] = (
        0.45 * engineered["match_rate"].clip(0, 1)
        + 0.25 * _bounded(matches, 40)
        + 0.15 * engineered["selectivity_balance"]
        + 0.15 * _bounded(engineered["social_pull"], 50)
    )
    engineered["conversation_quality"] = (
        0.40 * _bounded(engineered["msg_per_match"], 10)
        + 0.30 * _bounded(messages, 80)
        + 0.20 * _bounded(emoji_rate, 1)
        + 0.10 * _bounded(app_usage, 300)
    )
    engineered["profile_quality"] = (
        0.60 * engineered["profile_completeness"]
        + 0.25 * _bounded(bio_length, 300)
        + 0.15 * _bounded(profile_pics, 6)
    )
    engineered["connection_score"] = (
        0.35 * engineered["match_quality"]
        + 0.30 * engineered["conversation_quality"]
        + 0.20 * engineered["profile_quality"]
        + 0.15 * _bounded(app_usage, 300)
        - 0.10 * _bounded(engineered["swipe_excess"], 0.30)
    )
    engineered["browser_issue"] = (
        0.45 * (1 - _bounded(app_usage, 300))
        + 0.35 * (1 - _bounded(messages, 80))
        + 0.20 * (1 - _bounded(matches, 40))
    )
    engineered["swipe_issue"] = 0.55 * _bounded(
        engineered["swipe_excess"], 0.30
    ) + 0.45 * (1 - engineered["match_rate"].clip(0, 1))

    return engineered


def construct_connection_stage(frame):
    """Create plain-language connection-readiness labels from funnel signals."""
    scored = add_connection_features(frame)

    score_rank = _rank01(scored["connection_score"])
    browser_issue = scored["browser_issue"]
    swipe_issue = scored["swipe_issue"]

    labels = pd.Series("Ready To Chat", index=scored.index, dtype="object")
    labels[score_rank >= 0.80] = "Likely To Connect"
    labels[score_rank <= 0.20] = "Needs Profile Help"

    middle = labels.eq("Ready To Chat")
    labels[
        middle
        & (browser_issue >= browser_issue.quantile(0.62))
        & (browser_issue >= swipe_issue)
    ] = "Mostly Browsing"

    middle = labels.eq("Ready To Chat")
    labels[middle & (swipe_issue >= swipe_issue.quantile(0.50))] = "Swipes Too Freely"

    ordered = pd.Categorical(labels, categories=STAGE_LABELS, ordered=True)
    return pd.Series(ordered, index=labels.index).astype(str)


print("Connection scoring functions loaded.")
print(f"Target: {TARGET_COL}")
print(f"Stages: {STAGE_LABELS}")

## 5. Preprocessing and Target Construction

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- Load ---
df = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")

# --- Construct 5-class target ---
print(f"\nConstructing {TARGET_COL} target...")
df[TARGET_COL] = construct_connection_stage(df)
dist = df[TARGET_COL].value_counts().sort_index()
print(f"Connection-stage distribution:\n{dist.to_dict()}")

# --- Feature engineering ---
print("\nEngineering features...")
df = add_connection_features(df)

# Interest tags -> binary columns
tags_series = df["interest_tags"].fillna("").str.split(r",\s*")
all_tags = set()
for tag_list in tags_series:
    all_tags.update(tag_list)
all_tags.discard("")
sorted_tags = sorted(all_tags)
print(f"  Found {len(sorted_tags)} unique interest tags")

for tag in sorted_tags:
    df[f"tag_{tag}"] = tags_series.apply(lambda x: 1 if tag in x else 0)

# Drop redundant columns
cols_to_drop = [
    "interest_tags",
    "app_usage_time_label",
    "swipe_right_label",
    "relationship_intent",
    "match_outcome",
    "connection_score",
    "browser_issue",
    "swipe_issue",
    "engagement_score",
]
dropped = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=dropped)
print(f"  Dropped: {dropped}")

# --- Encode categoricals ---
print("\nEncoding categorical features...")
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

income_order = [
    "Very Low",
    "Low",
    "Lower-Middle",
    "Middle",
    "Upper-Middle",
    "High",
    "Very High",
]
education_order = [
    "No Formal Education",
    "High School",
    "Associate's",
    "Bachelor's",
    "Master's",
    "Postdoc",
    "PhD",
]

for col, order in [
    ("income_bracket", income_order),
    ("education_level", education_order),
]:
    if col in X.columns:
        mapping = {v: i for i, v in enumerate(order)}
        X[col] = X[col].map(mapping).fillna(-1).astype(int)
print("  Ordinal encoded: income_bracket, education_level")

remaining_cat = X.select_dtypes(include=["object"]).columns.tolist()
if remaining_cat:
    X = pd.get_dummies(X, columns=remaining_cat, drop_first=True)
    print(f"  One-hot encoded: {remaining_cat}")

print(f"  Final feature count: {X.shape[1]}")

# --- Encode target ---
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
print(f"  Target classes: {list(target_encoder.classes_)}")

# --- Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"  Train: {X_train.shape} | Test: {X_test.shape}")

# --- Scale ---
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

# --- Feature selection via RF importance ---
print("  Selecting features via RF importance...")
selector_rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
selector_rf.fit(X_train_scaled, y_train)

importances = selector_rf.feature_importances_
importance_df = pd.DataFrame(
    {"Feature": X_train_scaled.columns, "Importance": importances}
).sort_values("Importance", ascending=False)
importance_df["Cumulative"] = importance_df["Importance"].cumsum()
threshold = 0.95 * importance_df["Importance"].sum()
selected_features = importance_df[importance_df["Cumulative"] <= threshold][
    "Feature"
].tolist()
if len(selected_features) < 20:
    selected_features = importance_df.head(20)["Feature"].tolist()

X_train_final = X_train_scaled[selected_features]
X_test_final = X_test_scaled[selected_features]
print(f"  Selected {len(selected_features)} / {X_train_scaled.shape[1]} features")
print(f"  Train: {X_train_final.shape} | Test: {X_test_final.shape}")

# --- Save artifacts ---
OUT_DIR = Path("Preprocessed_Data_V2")
OUT_DIR.mkdir(exist_ok=True)
X_train_final.to_csv(OUT_DIR / "X_train_selected_unresampled.csv", index=False)
X_test_final.to_csv(OUT_DIR / "X_test_selected.csv", index=False)
pd.DataFrame(y_train, columns=[TARGET_COL]).to_csv(
    OUT_DIR / "y_train_original.csv", index=False
)
pd.DataFrame(y_test, columns=[TARGET_COL]).to_csv(OUT_DIR / "y_test.csv", index=False)
joblib.dump(scaler, OUT_DIR / "scaler.pkl")
joblib.dump(target_encoder, OUT_DIR / "target_encoder.pkl")
joblib.dump(selected_features, OUT_DIR / "selected_features.pkl")

# Plot top features
plt.figure(figsize=(10, 8))
sns.barplot(
    x="Importance",
    y="Feature",
    data=importance_df.head(20),
    palette="magma",
    hue="Feature",
    legend=False,
)
plt.title("Top 20 Feature Importances (Unbiased)", fontsize=14)
plt.tight_layout()
plt.show()

print("\nPreprocessing complete!")

## 6. Train 6 Models with 5-Fold CV

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def make_pipeline(model):
    """Wrap model in SMOTE-in-pipeline (leakage-free CV)."""
    return Pipeline([("smote", SMOTE(random_state=42)), ("model", model)])


models = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=2000, random_state=42)
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=300, max_depth=20, random_state=42, n_jobs=-1
        )
    ),
    "Gradient Boosting": make_pipeline(
        GradientBoostingClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
        )
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            eval_metric="mlogloss",
        )
    ),
    "LightGBM": make_pipeline(
        LGBMClassifier(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
    ),
    "CatBoost": make_pipeline(
        CatBoostClassifier(
            iterations=300, depth=6, learning_rate=0.1, random_state=42, verbose=0
        )
    ),
}

base_results = []

for name, pipeline in models.items():
    print(f"Training {name}...", end=" ")
    t0 = pd.Timestamp.now()

    cv_scores = cross_val_score(
        pipeline, X_train_final, y_train, cv=cv, scoring="accuracy", n_jobs=-1
    )
    pipeline.fit(X_train_final, y_train)
    y_pred = pipeline.predict(X_test_final)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted")
    elapsed = (pd.Timestamp.now() - t0).total_seconds()

    base_results.append(
        {
            "Model": name,
            "CV Accuracy (mean)": round(cv_scores.mean(), 4),
            "CV Accuracy (std)": round(cv_scores.std(), 4),
            "Test Accuracy": round(test_acc, 4),
            "Test F1 (weighted)": round(test_f1, 4),
            "Train Time (s)": round(elapsed, 1),
        }
    )
    print(
        f"CV: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f} | "
        f"Test: {test_acc:.4f} | F1: {test_f1:.4f} | {elapsed:.1f}s"
    )

base_results_df = pd.DataFrame(base_results).sort_values(
    "Test Accuracy", ascending=False
)
print("\n--- Base Model Comparison ---")
print(base_results_df.to_string(index=False))

## 7. Hyperparameter Tuning (Top 3)

In [ ]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

# Select top 3 by CV accuracy (not test accuracy — prevents leakage)
top3_names = (
    pd.DataFrame(base_results)
    .sort_values("CV Accuracy (mean)", ascending=False)
    .head(3)["Model"]
    .tolist()
)
print(f"Tuning top 3 (by CV): {top3_names}")

param_distributions = {
    "Random Forest": {
        "model__n_estimators": randint(200, 600),
        "model__max_depth": randint(10, 40),
        "model__min_samples_split": randint(2, 15),
        "model__min_samples_leaf": randint(1, 10),
    },
    "Gradient Boosting": {
        "model__n_estimators": randint(100, 400),
        "model__max_depth": randint(3, 10),
        "model__learning_rate": uniform(0.01, 0.29),
        "model__subsample": uniform(0.6, 0.4),
    },
    "XGBoost": {
        "model__n_estimators": randint(200, 600),
        "model__max_depth": randint(3, 10),
        "model__learning_rate": uniform(0.01, 0.29),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.6, 0.4),
        "model__reg_alpha": uniform(0.001, 9.999),
    },
    "LightGBM": {
        "model__n_estimators": randint(200, 600),
        "model__max_depth": randint(4, 12),
        "model__learning_rate": uniform(0.01, 0.29),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.6, 0.4),
    },
    "CatBoost": {
        "model__iterations": randint(200, 600),
        "model__depth": randint(4, 10),
        "model__learning_rate": uniform(0.01, 0.29),
        "model__l2_leaf_reg": uniform(0.001, 9.999),
    },
    "Logistic Regression": {
        "model__C": uniform(0.01, 99.99),
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"],
    },
}

tuned_results = {}
tuned_pipelines = {}

for name in top3_names:
    print(f"\nTuning {name}...", end=" ")
    t0 = pd.Timestamp.now()

    search = RandomizedSearchCV(
        models[name],
        param_distributions=param_distributions[name],
        n_iter=20,
        cv=3,
        scoring="accuracy",
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train_final, y_train)

    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test_final)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted")
    elapsed = (pd.Timestamp.now() - t0).total_seconds()

    tuned_results[name] = {
        "best_params": search.best_params_,
        "cv_score": round(search.best_score_, 4),
        "test_acc": round(test_acc, 4),
        "test_f1": round(test_f1, 4),
    }
    tuned_pipelines[name] = best_pipe
    print(
        f"CV: {search.best_score_:.4f} | Test: {test_acc:.4f} | "
        f"F1: {test_f1:.4f} | {elapsed:.1f}s | {search.best_params_}"
    )

print("\nTuning complete!")

## 8. Select Best Model, Calibrate, and Validate

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Find best by CV accuracy (not test accuracy)
all_candidates = []
for r in base_results:
    all_candidates.append(
        {
            "name": r["Model"],
            "selection_score": r["CV Accuracy (mean)"],
            "test_acc": r["Test Accuracy"],
            "source": "base",
        }
    )
for name, r in tuned_results.items():
    all_candidates.append(
        {
            "name": name,
            "selection_score": r["cv_score"],
            "test_acc": r["test_acc"],
            "source": "tuned",
        }
    )

best_candidate = max(all_candidates, key=lambda x: x["selection_score"])
best_name = best_candidate["name"]
print(
    f"Best model: {best_name} ({best_candidate['source']}) — "
    f"CV: {best_candidate['selection_score']:.4f} | "
    f"Test Acc: {best_candidate['test_acc']:.4f}"
)

# Get pipeline
if best_name in tuned_pipelines:
    best_pipeline = tuned_pipelines[best_name]
else:
    best_pipeline = models[best_name]

# Calibrate
base_estimator = best_pipeline.named_steps["model"]
calibrated_pipeline = Pipeline(
    [
        ("smote", SMOTE(random_state=42)),
        ("model", CalibratedClassifierCV(base_estimator, method="sigmoid", cv=3)),
    ]
)
calibrated_pipeline.fit(X_train_final, y_train)

y_pred_cal = calibrated_pipeline.predict(X_test_final)
y_proba_cal = calibrated_pipeline.predict_proba(X_test_final)
cal_acc = accuracy_score(y_test, y_pred_cal)
cal_f1 = f1_score(y_test, y_pred_cal, average="weighted")
print(f"Calibrated — Accuracy: {cal_acc:.4f}, F1: {cal_f1:.4f}")

# Calibration plot — one-vs-rest for all classes
n_classes = y_proba_cal.shape[1]
class_names_cal = [str(c) for c in target_encoder.classes_]

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfectly calibrated")
for cls_idx in range(n_classes):
    fraction_of_positives, mean_predicted_value = calibration_curve(
        (y_test == cls_idx).astype(int),
        y_proba_cal[:, cls_idx],
        n_bins=10,
        strategy="uniform",
    )
    plt.plot(
        mean_predicted_value,
        fraction_of_positives,
        "s-",
        label=f"Class {class_names_cal[cls_idx]}",
    )
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title(f"Calibration Plot — {best_name} (All Classes)")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. SHAP Interpretability

We train a separate Random Forest for SHAP TreeExplainer (consistent with the main pipeline).

In [ ]:
import shap

# Train a separate RF for SHAP (matches train.py approach)
shap_model = RandomForestClassifier(
    n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
)
shap_model.fit(X_train_final, y_train)

SHAP_SAMPLE_SIZE = 1000
shap_sample = X_train_final.sample(
    n=min(SHAP_SAMPLE_SIZE, len(X_train_final)), random_state=42
)
explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(shap_sample)

# For multi-class, use the first class or mean absolute
sv = shap_values[0] if isinstance(shap_values, list) else shap_values

# Beeswarm plot
print("SHAP Beeswarm Plot:")
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, shap_sample, show=False, max_display=20)
plt.title(f"SHAP Feature Importance — {best_name}", fontsize=13)
plt.tight_layout()
plt.show()

# Bar plot
print("\nSHAP Bar Plot:")
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, shap_sample, plot_type="bar", show=False, max_display=20)
plt.title(f"SHAP Mean |SHAP| — {best_name}", fontsize=13)
plt.tight_layout()
plt.show()

# Feature importance from the SHAP model
importances_shap = shap_model.feature_importances_
importance_shap_df = pd.DataFrame(
    {"Feature": X_train_final.columns, "Importance": importances_shap}
).sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    x="Importance",
    y="Feature",
    data=importance_shap_df.head(20),
    palette="magma",
    hue="Feature",
    legend=False,
)
plt.title("Top 20 Feature Importances (RF)", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Final Comparison and Classification Report

In [ ]:
# Build final comparison table
final_results = base_results.copy()

for name, r in tuned_results.items():
    final_results.append(
        {
            "Model": f"{name} (tuned)",
            "CV Accuracy (mean)": r["cv_score"],
            "CV Accuracy (std)": 0,
            "Test Accuracy": r["test_acc"],
            "Test F1 (weighted)": r["test_f1"],
            "Train Time (s)": 0,
        }
    )

final_results.append(
    {
        "Model": f"{best_name} (calibrated)",
        "CV Accuracy (mean)": 0,
        "CV Accuracy (std)": 0,
        "Test Accuracy": round(cal_acc, 4),
        "Test F1 (weighted)": round(cal_f1, 4),
        "Train Time (s)": 0,
    }
)

# Majority baseline
final_results.append(
    {
        "Model": "Majority Baseline",
        "CV Accuracy (mean)": round(max(np.bincount(y_train)) / len(y_train), 4),
        "CV Accuracy (std)": 0,
        "Test Accuracy": round(max(np.bincount(y_test)) / len(y_test), 4),
        "Test F1 (weighted)": 0,
        "Train Time (s)": 0,
    }
)

final_df = (
    pd.DataFrame(final_results)
    .sort_values("Test Accuracy", ascending=False)
    .reset_index(drop=True)
)

print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)
print(final_df.to_string(index=False))

# Classification report
y_pred_best = best_pipeline.predict(X_test_final)
class_names = [str(c) for c in target_encoder.classes_]
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (Best Model)")
print("=" * 70)
print(classification_report(y_test, y_pred_best, target_names=class_names))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    x="Test Accuracy",
    y="Model",
    data=final_df,
    palette="viridis",
    ax=axes[0],
    hue="Model",
    legend=False,
)
axes[0].set_title("Test Accuracy by Model", fontsize=14)
axes[0].set_xlim(final_df["Test Accuracy"].min() - 0.02, 1.0)

nonzero_f1 = final_df[final_df["Test F1 (weighted)"] > 0].copy()
if len(nonzero_f1) > 0:
    sns.barplot(
        x="Test F1 (weighted)",
        y="Model",
        data=nonzero_f1,
        palette="magma",
        ax=axes[1],
        hue="Model",
        legend=False,
    )
    axes[1].set_title("Test F1 (Weighted) by Model", fontsize=14)

plt.tight_layout()
plt.show()

print("\nPipeline complete! Best model:", best_name)

## 11. AutoML Baseline Comparison

We compare our manually tuned pipeline against automated baselines to demonstrate that our approach achieves competitive or superior performance while providing full control over preprocessing, feature engineering, and model interpretability.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

print("=" * 70)
print("COMPARISON: Manual Pipeline vs. Automated Baselines")
print("=" * 70)

# 1. Majority-class baseline (DummyClassifier)
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train_final, y_train)
y_dummy = dummy.predict(X_test_final)
dummy_acc = accuracy_score(y_test, y_dummy)
dummy_f1 = f1_score(y_test, y_dummy, average="weighted")
print(f"\n1. Majority-class baseline:  Acc={dummy_acc:.4f}, F1={dummy_f1:.4f}")

# 2. Random-guess baseline
dummy_uniform = DummyClassifier(strategy="uniform", random_state=42)
dummy_uniform.fit(X_train_final, y_train)
y_uniform = dummy_uniform.predict(X_test_final)
uniform_acc = accuracy_score(y_test, y_uniform)
uniform_f1 = f1_score(y_test, y_uniform, average="weighted")
print(f"2. Random-guess baseline:    Acc={uniform_acc:.4f}, F1={uniform_f1:.4f}")

# 3. Untuned best model (no hyperparameter search)
untuned_pipe = models[best_name]
untuned_pipe.fit(X_train_final, y_train)
y_untuned = untuned_pipe.predict(X_test_final)
untuned_acc = accuracy_score(y_test, y_untuned)
untuned_f1 = f1_score(y_test, y_untuned, average="weighted")
print(f"3. Untuned {best_name}:  Acc={untuned_acc:.4f}, F1={untuned_f1:.4f}")

# 4. Our tuned + calibrated model
print(f"4. Tuned + calibrated model: Acc={cal_acc:.4f}, F1={cal_f1:.4f}")

# 5. Best base model (no tuning)
best_base_acc = max(r["Test Accuracy"] for r in base_results)
print(f"5. Best base model (no tuning): Acc={best_base_acc:.4f}")

print("\n" + "=" * 70)
print("CONCLUSION")
print("=" * 70)
improvement_over_dummy = ((cal_acc - dummy_acc) / dummy_acc) * 100
print(
    f"Our tuned pipeline improves over the majority baseline by "
    f"{improvement_over_dummy:.1f}%."
)
print(
    f"Hyperparameter tuning improved accuracy from {untuned_acc:.4f} to {cal_acc:.4f}."
)
print(
    "Auto-sklearn (autoML) would automate model selection + tuning but "
    "sacrifices interpretability. Our manual pipeline provides SHAP "
    "explanations, calibration plots, and full control."
)

## 12. Save Artifacts

In [ ]:
RESULTS_DIR = Path("ML_Results")
RESULTS_DIR.mkdir(exist_ok=True)

joblib.dump(calibrated_pipeline, RESULTS_DIR / "best_tuned_model.pkl")
joblib.dump(target_encoder, RESULTS_DIR / "target_encoder.pkl")
joblib.dump(selected_features, RESULTS_DIR / "selected_features.pkl")
joblib.dump(scaler, RESULTS_DIR / "scaler.pkl")
final_df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)

# Save classification report
with open(RESULTS_DIR / "classification_report.txt", "w") as f:
    f.write(f"Best Model: {best_name}\n\n")
    f.write(classification_report(y_test, y_pred_best, target_names=class_names))

print("Artifacts saved to ML_Results/")
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"  {f.name}")

## 13. Download Artifacts (Optional)

Download the trained model and artifacts to use locally or deploy.

In [ ]:
# Uncomment to download artifacts
# import shutil
# shutil.make_archive("ML_Results", "zip", "ML_Results")
# shutil.make_archive("Preprocessed_Data_V2", "zip", "Preprocessed_Data_V2")
# from google.colab import files
# files.download("ML_Results.zip")
# files.download("Preprocessed_Data_V2.zip")